<a href="https://colab.research.google.com/github/Adjani09/M-dulo-2.-Actividad-did-ctica-2-M3/blob/main/simulacion_bancaria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np

# Semilla para garantizar la replicabilidad de los resultados exactos
random.seed(42)
np.random.seed(42)

# Definición de parámetros estocásticos (Probabilidades y tiempos medios)
# Perfil de Retiros (70% de probabilidad total)
retiro_probs = [0.23, 0.40, 0.17, 0.20]
retiro_arr = [1.0, 2.0, 3.0, 3.0]
retiro_ser = [1.0, 2.0, 3.0, 4.0]

# Consignacion (30%)
consignacion_probs = [0.10, 0.20, 0.30, 0.40]
consignacion_arr = [1.0, 2.0, 3.0, 4.0]
consignacion_ser = [3.0, 3.0, 5.0, 7.0]

# Cálculo del tiempo medio global de llegada ponderado
mean_arr_retiro = sum(p * t for p, t in zip(retiro_probs, retiro_arr)) # 2.14
mean_arr_cons = sum(p * t for p, t in zip(consignacion_probs, consignacion_arr)) # 3.0
global_mean_arrival = 0.70 * mean_arr_retiro + 0.30 * mean_arr_cons # 2.398

def run_simulation_replica(replica_id):
    #Simula una réplica de una jornada laboral de 8 horas (480 minutos).
    sim_time = 480.0
    current_time = 0.0

    # Estado de disponibilidad de los 3 cajeros (tiempo en el que quedan libres)
    teller_free_time = [0.0, 0.0, 0.0]

    # Almacenamiento de métricas por cajero
    # Listas para cada cajero [tiempo de servicio y tiempo de espera]
    teller_metrics = {0: [], 1: [], 2: []}

    # Contador de usuarios por subtipo dentro de la réplica
    # Tipos: 'Retiro_Rapido', 'Retiro_Normal', 'Retiro_Lento', 'Retiro_MuyLento',
    #        'Cons_Rapido', 'Cons_Normal', 'Cons_Lento', 'Cons_MuyLento'
    counts_by_type = {
        'Retiro_Rapido': 0, 'Retiro_Normal': 0, 'Retiro_Lento': 0, 'Retiro_MuyLento': 0,
        'Cons_Rapido': 0, 'Cons_Normal': 0, 'Cons_Lento': 0, 'Cons_MuyLento': 0
    }

    while True:
        # Generación del tiempo entre arribos (Distribución Exponencial)
        inter_arrival = np.random.exponential(global_mean_arrival)
        current_time += inter_arrival

        # Condición de parada: finalizar si supera las 8 horas
        if current_time > sim_time:
            break

        # Determinación probabilística del tipo de usuario
        if random.random() < 0.70:
            # Flujo de retiros
            r = random.random()
            if r < 0.23:
                sub_type = 'Retiro_Rapido'
                mean_ser = 1.0
            elif r < 0.23 + 0.40:
                sub_type = 'Retiro_Normal'
                mean_ser = 2.0
            elif r < 0.23 + 0.40 + 0.17:
                sub_type = 'Retiro_Lento'
                mean_ser = 3.0
            else:
                sub_type = 'Retiro_MuyLento'
                mean_ser = 4.0
        else:
            # Flujo de consignaciones
            r = random.random()
            if r < 0.10:
                sub_type = 'Cons_Rapido'
                mean_ser = 3.0
            elif r < 0.10 + 0.20:
                sub_type = 'Cons_Normal'
                mean_ser = 3.0
            elif r < 0.10 + 0.20 + 0.30:
                sub_type = 'Cons_Lento'
                mean_ser = 5.0
            else:
                sub_type = 'Cons_MuyLento'
                mean_ser = 7.0

        #conteos x tipo y subtipo
        counts_by_type[sub_type] += 1

        # Generación del tiempo de servicio (Distribución Exponencial)
        service_time = np.random.exponential(mean_ser)

        # Asignación aleatoria y uniforme a uno de los 3 cajeros (0, 1 o 2)
        assigned_teller = random.randint(0, 2)

        # Cálculo de tiempos de espera en cola
        arrival_time = current_time
        start_service_time = max(arrival_time, teller_free_time[assigned_teller])
        wait_time = start_service_time - arrival_time

        # Actualización del estado del cajero asignado
        teller_free_time[assigned_teller] = start_service_time + service_time

        # Guardar registros de la transacción
        teller_metrics[assigned_teller].append({
            'type': sub_type,
            'service_time': service_time,
            'wait_time': wait_time
        })

    return counts_by_type, teller_metrics

# Ejecución de las 10 réplicas
replicas_counts = []
replicas_teller_metrics = []

for i in range(10):
    counts, metrics = run_simulation_replica(i)
    replicas_counts.append(counts)
    replicas_teller_metrics.append(metrics)

# Procesa los resultados
print("--- SUMMARY OF REPLICAS ---")
for i in range(10):
    total_users_rep = sum(replicas_counts[i].values())
    print(f"Replica {i+1}: Total Users = {total_users_rep}, {replicas_counts[i]}")

# 1. Cajero analisis
teller_avg_service = {0: [], 1: [], 2: []}
teller_avg_wait = {0: [], 1: [], 2: []}

for i in range(10):
    for t in range(3):
        services = [m['service_time'] for m in replicas_teller_metrics[i][t]]
        waits = [m['wait_time'] for m in replicas_teller_metrics[i][t]]
        if services:
            teller_avg_service[t].append(np.mean(services))
            teller_avg_wait[t].append(np.mean(waits))

print("\n--- MÉTRICAS DE CAJERO (PROMEDIO ENTRE RÉPLICAS) ---")
for t in range(3):
    print(f"Cajero {t+1}: Tiempo medio de servicio = {np.mean(teller_avg_service[t]):.3f} min, Tiempo medio de espera = {np.mean(teller_avg_wait[t]):.3f} min")

# 2. Usuarios promedio de cada categoría en todas las réplicas
print("\n--- MÉTRICAS DE CATEGORÍA (PROMEDIO POR RÉPLICA) ---")
categories = list(replicas_counts[0].keys())
total_counts_all = {cat: 0 for cat in categories}
for i in range(10):
    for cat in categories:
        total_counts_all[cat] += replicas_counts[i][cat]

for cat in categories:
    print(f"{cat}: Average Users = {total_counts_all[cat]/10:.1f}, Total Users = {total_counts_all[cat]}")

# 3. Encuentra qué réplica tuvo el mínimo de usuarios para cada categoría general o en general.
replica_totals = [sum(replicas_counts[i].values()) for i in range(10)]
print(f"\nTotal de usuarios atendidos en cada réplica: {replica_totals}")
min_replica_idx = np.argmin(replica_totals)
print(f"Cantidad de réplicas evaluadas: {len(replica_totals)}")
print(f"Réplica con el menor número total de usuarios: Replica {min_replica_idx+1}")

--- SUMMARY OF REPLICAS ---
Replica 1: Total Users = 203, {'Retiro_Rapido': 33, 'Retiro_Normal': 54, 'Retiro_Lento': 29, 'Retiro_MuyLento': 34, 'Cons_Rapido': 9, 'Cons_Normal': 9, 'Cons_Lento': 13, 'Cons_MuyLento': 22}
Replica 2: Total Users = 174, {'Retiro_Rapido': 26, 'Retiro_Normal': 43, 'Retiro_Lento': 24, 'Retiro_MuyLento': 29, 'Cons_Rapido': 6, 'Cons_Normal': 13, 'Cons_Lento': 17, 'Cons_MuyLento': 16}
Replica 3: Total Users = 199, {'Retiro_Rapido': 33, 'Retiro_Normal': 53, 'Retiro_Lento': 38, 'Retiro_MuyLento': 22, 'Cons_Rapido': 6, 'Cons_Normal': 8, 'Cons_Lento': 18, 'Cons_MuyLento': 21}
Replica 4: Total Users = 204, {'Retiro_Rapido': 28, 'Retiro_Normal': 58, 'Retiro_Lento': 22, 'Retiro_MuyLento': 37, 'Cons_Rapido': 6, 'Cons_Normal': 11, 'Cons_Lento': 18, 'Cons_MuyLento': 24}
Replica 5: Total Users = 195, {'Retiro_Rapido': 25, 'Retiro_Normal': 58, 'Retiro_Lento': 30, 'Retiro_MuyLento': 38, 'Cons_Rapido': 2, 'Cons_Normal': 6, 'Cons_Lento': 16, 'Cons_MuyLento': 20}
Replica 6: Tota